# qNLP Experiment 1

* Python v. 3.9
* discopy v. 1.4.1

In [1]:
!date 

Wed Sep 13 06:52:41 EDT 2023


In [2]:
# !pip install discopy
# !pip install pytket
# !pip install pytket-qiskit
# !pip install noisyopt

In [3]:
!python --version 

Python 3.9.17


In [4]:
# !pip install discopy

In [6]:
import numpy as np
from time import time

# from discopy import CircuitFunctor
# from discopy import Ty, Word
from discopy.closed import Ty
# from discopy.closed import Word
from discopy.grammar.pregroup import Word


from discopy.grammar.pregroup import brute_force
from discopy.grammar.pregroup import draw
from discopy.quantum.circuit import Circuit
from discopy.quantum.gates import sqrt, Ket, H, Rx, CX, SWAP

ImportError: cannot import name 'draw' from 'discopy.grammar.pregroup' (/Users/saguinaga/opt/anaconda3/envs/pyq39/lib/python3.9/site-packages/discopy/grammar/pregroup.py)

In [ ]:
from pytket.extensions.qiskit import AerBackend, IBMQBackend
from pytket.extensions.qiskit import tk_to_qiskit

from qiskit import IBMQ
from qiskit.providers.aer import noise

In [ ]:
s, n = Ty('s'), Ty('n')
Alice = Word('Alice', n)
loves = Word('loves', n.r @ s @ n.l)
Bob =  Word('Bob', n)
who = Word('who', n.r @ n @ s.l @ n)
is_rich = Word('is rich', n.r @ s)

vocab = [Alice, loves, Bob, who, is_rich]

In [ ]:
gen, n_sentences = brute_force(*vocab), 10
sentences, parsing = list(), dict()

print("Brute force search for grammatical sentences:")

start = time()
for i in range(n_sentences):
    diagram = next(gen)
    sentence = ' '.join(str(w)
        for w in diagram.boxes if isinstance(w, Word)) + '.'
    sentences.append(sentence)
    parsing.update({sentence: diagram})
    print(sentence)

print("\n{:.2f} seconds to generate {} sentences.".format(time() - start, n_sentences))

In [ ]:
print("Diagram for 'Alice who loves Bob is rich.':")
draw(parsing['Alice who loves Bob is rich.'], draw_types=False)

In [ ]:
GHZ = sqrt(2) @ Ket(0, 0, 0)\
    >> Circuit.id(1) @ H @ Circuit.id(1)\
    >> Circuit.id(1) @ CX\
    >> (SWAP >>  CX) @ Circuit.id(1)

print("Circuit for the GHZ state, used as ansatz for \"who\":")
GHZ.draw(aspect='auto', draw_types=False)

In [ ]:
def intransitive_ansatz(phase):
    return Ket(0) >> Rx(phase)

def transitive_ansatz(phase):
    return sqrt(2) @ Ket(0, 0) >> H @ Rx(phase) >> CX

F = lambda params: CircuitFunctor(
    ob={s: 0, n: 1},
    ar={Alice: Ket(0),
        loves: transitive_ansatz(params[0]),
        Bob: Ket(1),
        who: GHZ,
        is_rich: intransitive_ansatz(params[1])}
)

params0 = np.array([0.5, 1.])
print("Circuit for 'Alice who loves Bob is rich.':")
circuit = F(params0)(parsing['Alice who loves Bob is rich.'])
circuit.draw(figsize=(10, 8), aspect='auto', draw_types=False)

In [ ]:
tk_circ = circuit.to_tk()
print("{}:\n{}\n".format(tk_circ, '\n'.join(map(str, tk_circ))))
print("post selection:\n{}\n".format(tk_circ.post_selection))
print("scalar:\n{}\n".format(tk_circ.scalar))
print("qiskit circuit:")
tk_to_qiskit(tk_circ).draw()

In [ ]:
token = 'rovoked-token-from-ibm'
IBMQ.save_account(token)
IBMQ.load_account()
provider = IBMQ.get_provider(hub='ibm-q-company',group='internal',project='company-us-quan')

In [ ]:
def evaluate(params, sentences, backend=AerBackend(), n_shots=2**10, seed=0):
    circuits = [F(params)(parsing[s]) for s in sentences]
    results = [Circuit.eval(
                circuit,
                backend=backend,
                n_shots=n_shots,
                seed=seed,
                compilation=backend.default_compilation_pass(2)) for circuit in circuits]
    tensors = [np.abs(result.array) for result in results]
    return tensors

In [ ]:
start = time()
corpus = dict(zip(sentences, evaluate(params0, sentences, backend=AerBackend())))
print("{:.3f} seconds to compute the corpus.\n".format(time() - start))

delta = .1

print("True sentences:\n{}\n".format('\n'.join("{} ({:.3f})".format(sentence, scalar)
    for sentence, scalar in corpus.items() if scalar > .5 + delta)))
print("False sentences:\n{}\n".format('\n'.join("{} ({:.3f})".format(sentence, scalar)
    for sentence, scalar in corpus.items() if scalar < .5 - delta)))
print("Maybe sentences:\n{}".format('\n'.join("{} ({:.3f})".format(sentence, scalar)
    for sentence, scalar in corpus.items() if .5 - delta <= scalar <= .5 + delta)))

## Classical Methods

In [1]:
import noisyopt

from sklearn.model_selection import train_test_split
from random import random, seed; 

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
sentence_train, sentence_test = train_test_split(sentences, test_size=0.5, random_state=237)

print("Training set:\n{}\n".format('\n'.join(sentence_train)))
print("Testing set:\n{}".format('\n'.join(map(str, sentence_test))))

In [ ]:
def loss(params, sentences=sentence_train):
    return - np.mean(np.array([
        (corpus[sentence] - scalar) ** 2
        for sentence, scalar in zip(sentences, evaluate(params, sentences))]))

In [ ]:
seed(337)

params = np.array([random(), random()])

print("Random initialisation: {}".format(params))

print("\nIs Alice who loves Bob rich?")
print("Yes, she is."
      if evaluate(params, ['Alice who loves Bob is rich.'])[0] > .5 + delta
      else "No, she isn't.")

In [ ]:
i, start = 0, time()

def callback(params):
    global i
    i += 1
    print("Epoch {} ({:.0f} seconds since start): {}".format(i, time() - start, params))

result = noisyopt.minimizeSPSA(
    loss, params, paired=False, callback=callback, niter=10, a=2)
result

In [ ]:
print("Testing loss: {}".format(loss(result.x, sentences=sentence_test)))

In [ ]:
print("Is Alice who loves Bob rich?")
print("Yes, she is."
      if evaluate(result.x, ['Alice who loves Bob is rich.'])[0] > .5 + delta
      else "No, she isn't.")